# 03 How Tool Calling Really Works

## 工具调用不是外挂，而是受控决策的结构化出口

只要开始做 LLM 系统，几乎一定会碰到一个很有迷惑性的说法：模型会调用工具。这个说法在产品演示里没问题，但如果直接拿它来理解系统结构，会很快走偏。

模型并不会像程序线程那样主动执行函数。它真正做的事情，是在当前上下文下输出一段被系统解释为“调用某个工具”的结构化意图。真正完成执行、返回结果、管理失败、继续推进流程的，始终是外部 runtime。

所以，这一章不把 tool calling 当成一个 API 功能点，而把它当成一个更本质的问题来讨论：**自然语言决策，是如何被压缩成可执行系统动作的。**

## 先给结论

如果只保留一句最核心的话，那就是：

> Tool calling 的本质，不是模型获得了执行能力，而是模型被要求在某些场景下，把下一步输出收缩为一个结构化动作选择。

这句话拆开看，有四层含义：

- 模型首先要判断当前问题是否需要外部能力
- 一旦需要，模型要在候选工具之间做选择
- 选择之后，模型还要生成足够可执行的参数结构
- 外部系统再把这个结构化输出解释成真实调用，并将结果送回上下文

因此，tool calling 不是一个点状能力，而是一条完整链路。很多系统的问题，不是出在“函数没写好”，而是出在这条链路里某一环被想得过于简单。

## 1. 模型为什么会选择工具，而不是继续直接回答

这个问题看起来简单，实际上决定了 tool calling 的全部上限。因为工具并不会自己跳出来，模型也不会天然认为“有工具就该调用工具”。

模型会不会走工具路径，取决于它在当前上下文里学到的判断：

- 这个问题能不能仅靠已有上下文回答
- 如果直接回答，是否违反 system prompt 的边界
- 某个工具的描述是否看起来比自然语言回答更贴近当前目标
- 输出一个结构化调用是否比继续自由生成更符合当前任务分布

所以工具调用从来不是“发现缺信息就自动触发”，而是一次概率决策。也正因为它是决策而不是硬编码规则，系统才必须在 schema、prompt 和 runtime 三层同时施加约束。

## 2. Tool Schema 的作用，不是描述函数，而是塑造选择空间

从程序员视角看，tool schema 像是在给函数写签名；但从模型视角看，schema 更像是在定义一个受限输出空间。

当模型看到一个工具时，它真正接收到的不是“这里有个 Python 函数”，而是一组带语义暗示的选择条件：

- 这个工具叫什么
- 这个名字像不像当前任务需要的动作
- 这个工具的描述是否清楚说明了适用场景
- 这个参数结构是否容易与当前问题映射起来

这就是为什么工具定义的好坏，常常直接决定工具是否被正确选中。因为 schema 不只是给程序看的契约，也是给模型看的决策提示。

一个更像样的 schema，通常会把动作意义、适用场景和参数语义都说清楚。比如：

- `get_weather_forecast`
- 用途：获取指定城市在指定日期的天气预报，用于天气、出行或户外活动判断
- 参数：`location`、`date`

而一个弱 schema 往往长这样：

- `handle_data`
- 描述：处理数据
- 参数：`input`

两者都能被程序接住，但前者给模型提供的是动作语义，后者提供的只是形式合法性。

上面两个 schema 在程序层都能成立，但对模型而言，差别非常大。

第一个 schema 把动作、适用场景和参数意义都说得比较清楚，模型更容易判断“这是不是我现在应该选的工具”。第二个 schema 虽然形式合法，但几乎不给模型任何语义抓手。它当然还能被调，但更容易被误选、漏选或传错参数。

所以工具 schema 的设计质量，本质上不是“代码整洁度”问题，而是“模型决策清晰度”问题。

## 3. 工具选择是概率决策，不是规则匹配

工程上最容易产生误判的一点，就是把工具选择理解成“命中关键词就调用对应函数”。真实情况远不是这样。

模型选择工具时，实际上在比较多个可能出口：

- 直接回答
- 先澄清问题
- 调用工具 A
- 调用工具 B
- 拒绝回答

它最终选哪一个，受很多因素共同影响：

- system prompt 是否强烈要求先验证
- 工具名字和描述是否贴近当前问题
- 当前问题是否容易被某个参数结构承接
- 历史轮次里是否已经有过类似决策模式

因此，工具选择的正确率不是单靠 schema 就能保证的，它本质上是模型在受控上下文里的决策质量问题。

## 4. 真正难的往往不是选工具，而是生成可执行参数

很多 demo 会把关注点放在“模型会不会选对工具”，但在实际系统里，更常出问题的是参数层。

因为一旦模型决定走工具路径，它还必须完成另一个任务：把自然语言问题映射为程序可执行的参数对象。这一步里会出现很多典型错误：

- 漏填必填字段
- 把语义相近但格式不合规的值传进去
- 自作主张补出用户没给的信息
- 参数名字对了，但值的粒度不对

这也是为什么好的 tool calling 系统几乎一定会在 runtime 层加入参数校验和失败回填，而不是把模型第一次输出当成天然正确的调用请求。

例如，用户说“帮我看下这周五上海适不适合拍外景”，模型可能会生成这样的调用意图：

- 工具：`get_weather_forecast`
- 参数：`location=上海`，`date=this_friday`

问题在于，这个调用即使在 schema 层面通过，也未必足够执行。因为 `this_friday` 这种值也许形式合法，但业务侧仍然需要一个具体日期。也就是说，schema 通过不等于业务就绪。

这个例子故意展示了一种很真实的情况：参数可能“形式合法”，但仍然“不足以执行”。

也就是说，tool calling 不只需要 schema 级校验，还往往需要业务级校验。前者回答的是“结构像不像”，后者回答的是“到底能不能拿去跑”。如果这两层不分开，系统就会在一个看似成功的调用上埋下隐患。

## 5. Tool Calling 真正成立，依赖一个外部闭环

模型输出工具调用意图，只是链路的中点，不是终点。一个完整的 tool calling 至少包含六步：

1. 模型读取当前消息和可用工具定义
2. 模型判断当前轮是否需要外部能力
3. 模型输出工具名和参数结构
4. Runtime 校验并执行对应工具
5. 工具结果以 `tool` 或等价角色回填上下文
6. 模型基于新上下文继续推理并收束为最终回答

如果少了后四步，tool calling 只是一个看起来很像动作的字符串；如果少了前两步，系统就会变成机械触发器，而不是受控决策系统。真正有价值的，是这六步连起来之后形成的执行闭环。

把这条闭环直接写成步骤，会比伪代码更清楚：

1. 模型读取当前消息和 tool schema。
2. 模型输出工具调用意图。
3. Runtime 校验这个意图。
4. Runtime 执行外部工具。
5. Runtime 把工具结果回填进上下文。
6. 模型基于新上下文综合出最终回答。

真正有价值的 tool calling，从来不是某个函数被叫起来，而是这六步形成的闭环。

## 6. Tool Result 为什么会改变后续推理

工具调用的真正价值，不只是“拿到外部数据”，而是它会重写模型接下来所处的上下文。

在工具结果返回之前，模型面对的是一个信息不足的问题；在工具结果返回之后，模型面对的是一个已经带有外部证据的问题。上下文一变，后续最优输出分布也会跟着变。

这意味着 tool result 不只是数据补丁，它还是一次决策环境更新。模型后面之所以更有可能给出更稳的答案，不是因为它突然变聪明了，而是因为它已经不再需要凭空补全关键事实。

## 7. 常见失败模式，不是偶发现象，而是结构性后果

只要把 tool calling 理解成一条链路，就会发现失败几乎可以出现在每一环：

- 模型本该调用工具却直接猜
- 模型选错工具
- 模型选对工具但参数不完整
- 参数形式通过但业务语义错误
- 工具返回噪音太多，反而污染后续推理
- 模型拿到结果后仍然忽略关键事实

这些问题并不说明 tool calling 没价值，反而说明它本质上是一个需要治理的系统接口，而不是“把函数挂上去就行”的快捷能力。

## 8. Tool Calling 还不是 Agent，但已经把动作接口建起来了

到这一章为止，模型已经具备了一种很关键的能力：它可以在某些上下文里把下一步输出压缩为一个结构化动作意图，并把这个意图交给外部系统处理。

但这仍然不等于 Agent 已经完整成立。因为 Agent 还需要更高一层的东西：

- 目标在多步流程中的持续存在
- 多次工具或资源访问的编排
- 中间状态的保留与更新
- 对失败的恢复与重新决策

可以把 tool calling 理解为 Agent 的动作接口层。它把“下一步要做什么”从自然语言里提炼出来，但真正决定“什么时候做、做几次、做到什么程度停止”的，仍然是后面的 Agent Runtime。

## 9. 本章结论

这一章可以压缩成六个判断：

- tool calling 的本质是结构化动作选择，不是模型直接执行函数。
- tool schema 既是程序契约，也是模型决策提示。
- 工具选择是概率决策，不是关键词命中。
- 参数生成往往比工具选择本身更脆弱。
- tool result 的价值不只是提供数据，而是重写后续推理环境。
- 真正有价值的不是某次“成功调用”，而是整个调用闭环被稳定组织起来。

下一章进入 Agent。本质问题会再上一个层级：当模型已经能输出动作意图，谁来维护目标、编排状态、组织多步循环，并决定这些工具到底该如何被连续使用。